Supplementary Code S2 — Model benchmarking using leakage-safe nested cross-validation

This notebook performs systematic benchmarking of candidate machine-learning classifiers on the cleaned myocardial infarction dataset using a rigorously leakage-controlled nested cross-validation framework. The objective is to obtain unbiased estimates of discriminative and probabilistic performance while transparently comparing multiple algorithms across predefined feature-set variants representing distinct clinical and translational scenarios.

All analyses are conducted exclusively on the cleaned dataset (df_clean.parquet) generated in Supplementary Code S1. Feature-set definitions (FULL, CLINICAL, BIOMARKERS) are loaded from governance artifacts (feature_variants.json) and treated as fixed, immutable inputs. No outcome-driven feature selection, filtering, or transformation is performed prior to cross-validation.

Experimental configuration (frozen snapshot)

Task: STEMI vs NSTEMI classification

Positive label: STEMI

Sample size: n = 57

Nested cross-validation:

Outer splits: 5 (stratified)

Inner splits: 5 (stratified)

Random state: 42

Primary scoring metric: ROC AUC

All downstream analyses use out-of-fold (OOF) predictions only.

No model is refit on the full dataset for performance reporting.

No separate hold-out test set is used.

Environment snapshot:

Python 3.12.12

NumPy 2.0.2

pandas 2.2.2

scikit-learn 1.6.1

package_versions

This configuration is serialized in S2_config_snapshot.json for reproducibility.

Methodological framework
Nested cross-validation design

Performance estimation follows stratified nested cross-validation:

The outer loop (5 folds) is used exclusively for unbiased performance estimation.

The inner loop (5 folds) performs hyperparameter optimization using grid search.

Hyperparameter tuning is fully isolated from outer test folds.

No post-selection refitting on the entire dataset is performed.

Performance metrics are computed only on outer-fold test partitions.

This design ensures strict separation between model selection and evaluation and minimizes optimistic bias.

Leakage-safe preprocessing

All preprocessing is implemented inside scikit-learn pipelines and fitted exclusively on training data within each fold:

Median imputation for numeric variables.

Standardization of numeric features.

Imputation and one-hot encoding for categorical variables.

No test-fold statistics are ever used to inform preprocessing decisions.

Feature-set governance and leakage firewall

Feature inclusion is governed by a layered, rule-based framework designed to prevent both explicit and subtle forms of information leakage.

1. S1 governance inheritance

Feature roles defined in S1 (clinical, biomarker, exclude, target, id) are imported and treated as immutable inputs.

Final feature-set sizes after all filtering:

FULL: 69 features

CLINICAL: 58 features

BIOMARKERS: 10 features

2. S2 conservative firewall (name-based exclusion)

An additional exclusion layer removes any variables whose names match predefined high-risk patterns, including:

Outcome labels or disease subtypes (e.g., STEMI, NSTEMI).

Terms indicating outcome, endpoint, target, or infarction.

Post-index events (follow-up, rehospitalization, death).

Procedural or treatment variables (PCI, CABG, stent, heparin, thrombolysis).

Temporal indicators (date, timestamp).

Peak, maximum, or summary values likely computed after presentation.

These patterns are specified explicitly via regular expressions and serialized in the configuration snapshot.

3. Missingness and constancy screening

Features are automatically removed if they:

Are entirely missing in the modeling cohort.

Exceed a missingness threshold of >95%.

Are constant.

Are near-constant (≥99.5% identical values).

All removals are logged in:

S2_all_missing_drop_audit.csv

S2_fold_allmissing_drop_audit.csv

4. Univariate biomarker leakage triage

To mitigate residual leakage risk specific to biomarker variables, a conservative univariate diagnostic step is applied only to the BIOMARKERS variant:

Each biomarker is evaluated using cross-validated univariate ROC AUC.

Biomarkers with AUC ≥ 0.98 are flagged as potentially leakage-like.

Flagged variables are blacklisted and excluded from multivariable modeling.

This step is strictly preventive and diagnostic. It is not a feature-selection strategy and does not aim to optimize predictive performance.

Blacklisted biomarkers are recorded in:

S2_biomarkers_blacklist.csv

Candidate models

Each feature-set variant is evaluated using the same candidate classifiers:

Penalized logistic regression (L1 and L2 penalties).

Linear support vector machine.

Random forest classifier.

Histogram-based gradient boosting classifier.

All models are trained and evaluated under identical cross-validation splits and preprocessing pipelines to ensure fair comparison.

Performance metrics

All performance metrics are computed exclusively on outer-fold test partitions.

Primary metric:

ROC AUC

Secondary metrics:

Area under the precision–recall curve (AUPRC)

Brier score

Accuracy

F1 score

Matthews correlation coefficient (MCC)

Fold-level metrics are retained to characterize variability.

Bootstrap confidence intervals for the winning model are computed using OOF predictions.

Out-of-fold predictions (single source of truth)

For every model and feature-set variant, out-of-fold predicted probabilities are stored.

OOF predictions constitute the sole input for:

Calibration analysis

Decision-curve analysis

Bootstrap confidence intervals

ROC and PR curve visualization

Locked-model evaluation (S9)

Permutation diagnostics (S10)

Workflow validation and robustness analyses (S11)

At no stage are in-sample predictions used for evaluation or inference.

Outputs (written to RESULTS_DIR)

The notebook generates the following auditable artifacts:

nestedcv_results_all_variants.csv — outer-fold performance metrics.

nestedcv_oof_probs_all_variants.csv — OOF predicted probabilities.

nestedcv_bestparams_all_variants.csv — inner-loop hyperparameters.

winner_model_by_variant.csv — benchmark winner per variant.

features_used_*.csv — final post-governance feature lists.

S2_leakage_sanity_flags.csv

S2_leakage_like_perf_flags.csv

S2_biomarkers_blacklist.csv

S2_governance_summary.csv

S2_governance_breakdown.csv

S2_all_missing_drop_audit.csv

outer_folds_indices.csv

S2_config_snapshot.json

package_versions.json

Together, these outputs establish a fully auditable, leakage-safe modeling foundation that supports all downstream calibration, interpretability, permutation, and robustness analyses (Supplementary Codes S3–S11).

## Imports, paths, config

In [1]:
from __future__ import annotations

import os
import json
import re
import warnings
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd

from sklearn.exceptions import ConvergenceWarning
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    accuracy_score,
    f1_score,
    matthews_corrcoef,
)
from sklearn.calibration import calibration_curve

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

warnings.filterwarnings("ignore", category=ConvergenceWarning)

ROOT = Path("/content")
PROCESSED_DIR = ROOT / "processed"
OUT_DIR = ROOT / "results" / "S2_nestedcv"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DF_PATH = PROCESSED_DIR / "df_clean.parquet"
DD_PATH = PROCESSED_DIR / "data_dictionary.csv"
VARIANTS_PATH_S1 = PROCESSED_DIR / "feature_variants.json"

CFG_SNAPSHOT = OUT_DIR / "S2_config_snapshot.json"
LEAK_FLAGS = OUT_DIR / "S2_leakage_sanity_flags.csv"
GOV_SUMMARY = OUT_DIR / "S2_governance_summary.csv"
ALLMISSING_AUDIT = OUT_DIR / "S2_all_missing_drop_audit.csv"
FOLD_DROP_AUDIT = OUT_DIR / "S2_fold_allmissing_drop_audit.csv"

TRIAGE_PATH = OUT_DIR / "S2_biomarkers_univariate_triage.csv"
BIO_BLACKLIST_PATH = OUT_DIR / "S2_biomarkers_blacklist.csv"

print("DF_PATH:", DF_PATH, "| exists:", DF_PATH.exists())
print("DD_PATH:", DD_PATH, "| exists:", DD_PATH.exists())
print("VARIANTS_PATH_S1:", VARIANTS_PATH_S1, "| exists:", VARIANTS_PATH_S1.exists())
print("OUT_DIR:", OUT_DIR.resolve())


DF_PATH: /content/processed/df_clean.parquet | exists: False
DD_PATH: /content/processed/data_dictionary.csv | exists: False
VARIANTS_PATH_S1: /content/processed/feature_variants.json | exists: False
OUT_DIR: /content/results/S2_nestedcv


## Configuration

In [2]:
@dataclass(frozen=True)
class Config:
    target_col: str = "typ_zawalu"


    task_labels_allowed: Tuple[str, ...] = ("STEMI", "NSTEMI")
    positive_label: str = "STEMI"

    outer_splits: int = 5
    inner_splits: int = 5
    random_state: int = 42

    scoring: str = "roc_auc"
    dca_thresholds: Tuple[float, ...] = tuple(np.linspace(0.01, 0.99, 99))


    hard_exclude_patterns: Tuple[str, ...] = (
        r"(_|^)(max|maks|peak|szczyt|highest|top)(_|\b|$)",
        r"(^|_)data(_|$)|(^|_)date(_|$)|(^|_)czas(_|$)|timestamp",
        r"wypis|follow|kontrol|rehosp|readmission|zgon|mort",
        r"procedur|pci|cabg|stent|angiograf",
        r"heparin|heparyn|trombol|statin|statyn|ticagrelor|prasugrel|clopidogrel|aspirin|\basa\b",
        r"\b(stemi|nstemi|\bua\b)\b",
        r"\b(outcome|endpoint|label|target)\b",
        r"\bzawal\b",
    )


    drop_if_missing_gt: float = 0.95


    triage_auc_threshold: float = 0.98
    triage_min_nonmissing: float = 0.20
    triage_min_unique: int = 3


    drop_constant_features: bool = True
    drop_near_constant_threshold: float = 0.995

CFG = Config()


## Definition of 4 models (LogReg, LinearSVM calibrated, RF, HGB)

In [3]:
MODEL_SPACES: Dict[str, Dict[str, Any]] = {
    "LogReg": {
        "estimator": LogisticRegression(
            solver="liblinear",
            max_iter=5000,
            class_weight="balanced",
            random_state=CFG.random_state,
        ),
        "param_grid": {
            "clf__C": [0.01, 0.1, 1.0, 10.0],
            "clf__penalty": ["l1", "l2"],
        },
    },

    "LinearSVM": {
        "estimator": SVC(
            kernel="linear",
            probability=True,
            class_weight="balanced",
            random_state=CFG.random_state,
        ),
        "param_grid": {"clf__C": [0.01, 0.1, 1.0, 10.0]},
    },

    "RF": {
        "estimator": RandomForestClassifier(
            n_estimators=500,
            random_state=CFG.random_state,
            class_weight="balanced_subsample",
            n_jobs=-1,
        ),
        "param_grid": {
            "clf__max_depth": [None, 3, 5, 8],
            "clf__min_samples_split": [2, 5, 10],
            "clf__min_samples_leaf": [1, 2, 4],
            "clf__max_features": ["sqrt", "log2", 0.5],
        },
    },

    "HGB": {
        "estimator": HistGradientBoostingClassifier(random_state=CFG.random_state),
        "param_grid": {
            "clf__learning_rate": [0.03, 0.1],
            "clf__max_depth": [2, 3],
            "clf__max_leaf_nodes": [15, 31],
            "clf__min_samples_leaf": [10, 20],
        },
    },
}

print("Models:", list(MODEL_SPACES.keys()))


Models: ['LogReg', 'LinearSVM', 'RF', 'HGB']


## Utils I/O + snapshot config

In [4]:
def save_json(obj: Any, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def read_parquet(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing input: {path}")
    return pd.read_parquet(path)

def read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing input: {path}")
    return pd.read_csv(path)

def get_package_versions() -> Dict[str, Any]:
    import sklearn, sys
    return {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "python": sys.version,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
    }

save_json(get_package_versions(), OUT_DIR / "package_versions.json")
save_json(asdict(CFG), CFG_SNAPSHOT)
print("Saved:", (OUT_DIR / "package_versions.json").name, "and", CFG_SNAPSHOT.name)


Saved: package_versions.json and S2_config_snapshot.json


## Loading data + feature variants

### Design note — why the **FULL** variant intentionally remains permissive

This notebook is part of a *methodological stress test* of leakage-aware workflows (see manuscript Section 5).
Accordingly, the **FULL** feature-set variant is intentionally defined as the **widest plausible predictor space available in the dataset** (subject only to minimal firewall rules such as removing the target itself and obvious outcome/procedure keywords).

**Interpretation guidance:**
- Near-perfect internal performance in **FULL** is treated as a **diagnostic signal** that motivates leakage scrutiny (e.g., downstream-coupled or post-baseline variables), **not** as evidence of deployable clinical accuracy.
- In contrast, **CLINICAL** and **BIOMARKERS** variants represent *more realistic availability assumptions* and serve as conservative comparators under the same nested-CV protocol.

Deliberately permissive* configuration used to reveal potential leakage patterns, rather than a recommended feature set for model development.

In [5]:
df = read_parquet(Path('/content') / "df_clean.parquet")
dd = read_csv(Path('/content') / "data_dictionary.csv")

VARIANTS_PATH_S1 = Path('/content') / "feature_variants.json"

if not VARIANTS_PATH_S1.exists():
    raise FileNotFoundError("Missing feature_variants.json from S1. Run S1 first.")

with open(VARIANTS_PATH_S1, "r", encoding="utf-8") as f:
    variants_s1 = json.load(f)

print("df shape:", df.shape)
print("variants sizes (S1):", {k: len(v) for k, v in variants_s1.items()})
print("data_dictionary roles:", dd["role"].value_counts(dropna=False).to_dict())


df shape: (152, 117)
variants sizes (S1): {'FULL': 73, 'CLINICAL': 62, 'BIOMARKERS': 11}
data_dictionary roles: {'clinical': 63, 'exclude': 39, 'biomarker': 10, 'id': 4, 'target': 1}


## Task definition (STEMI vs NSTEMI) + y

In [6]:
target = CFG.target_col
allowed = set(CFG.task_labels_allowed)

if target not in df.columns:
    raise ValueError(f"Target column '{target}' not found in df_clean.parquet")

df_task = df[df[target].isin(allowed)].copy()

print("Task class counts (STEMI vs NSTEMI):")
print(df_task[target].value_counts(dropna=False).to_string())

y = (df_task[target] == CFG.positive_label).astype("Int64")
if y.nunique() != 2:
    raise ValueError(f"Expected exactly 2 classes after filtering, got y.nunique()={y.nunique()}")

print("y distribution:", y.value_counts().to_dict())


Task class counts (STEMI vs NSTEMI):
typ_zawalu
NSTEMI    32
STEMI     25
y distribution: {np.int64(0): 32, np.int64(1): 25}


## Governance filter (role-based) + firewall filter (name-based)

In [7]:
def roles_allowed_features(dd: pd.DataFrame) -> set:

    allowed_roles = {"clinical", "biomarker"}
    return set(dd.loc[dd["role"].isin(allowed_roles), "column"].astype(str).tolist())

ALLOWED_BY_ROLE = roles_allowed_features(dd)

def passes_s2_firewall(feature: str, cfg: Config) -> Tuple[bool, str]:
    f = str(feature)
    if f == cfg.target_col:
        return False, "is_target"
    for pat in cfg.hard_exclude_patterns:
        if re.search(pat, f, flags=re.IGNORECASE):
            return False, f"pattern:{pat}"
    return True, ""

def apply_role_and_firewall_to_variants(
    variants: Dict[str, List[str]],
    df_cols: List[str],
    allowed_by_role: set,
    cfg: Config,
) -> Tuple[Dict[str, List[str]], pd.DataFrame]:
    rows = []
    out = {}
    for vname, feats in variants.items():
        kept = []
        for f in feats:
            if f not in df_cols:
                rows.append({"variant": vname, "feature": f, "status": "dropped", "reason": "not_in_df_task"})
                continue
            if f not in allowed_by_role:
                rows.append({"variant": vname, "feature": f, "status": "dropped", "reason": "role_not_predictor"})
                continue
            ok, reason = passes_s2_firewall(f, cfg)
            if ok:
                kept.append(f)
                rows.append({"variant": vname, "feature": f, "status": "kept", "reason": ""})
            else:
                rows.append({"variant": vname, "feature": f, "status": "dropped", "reason": reason})
        out[vname] = sorted(list(dict.fromkeys(kept)))
    return out, pd.DataFrame(rows)

variants_safe, firewall_audit = apply_role_and_firewall_to_variants(
    variants_s1,
    df_cols=list(df_task.columns),
    allowed_by_role=ALLOWED_BY_ROLE,
    cfg=CFG,
)
firewall_audit.to_csv(LEAK_FLAGS, index=False)

print("Variant sizes after role+firewall:", {k: len(v) for k, v in variants_safe.items()})
print("Saved:", LEAK_FLAGS.name)


Variant sizes after role+firewall: {'FULL': 73, 'CLINICAL': 62, 'BIOMARKERS': 11}
Saved: S2_leakage_sanity_flags.csv


## Drop all-missing / high-missing / constant / near-constant (global)

In [8]:
def drop_all_missing_high_missing_constant(
    df_task: pd.DataFrame,
    variants: Dict[str, List[str]],
    cfg: Config,
) -> Tuple[Dict[str, List[str]], pd.DataFrame]:
    rows = []
    out = {}
    for v, feats in variants.items():
        kept = []
        for f in feats:
            if f not in df_task.columns:
                rows.append({"variant": v, "feature": f, "action": "drop", "reason": "not_in_df_task"})
                continue

            nn = int(df_task[f].notna().sum())
            if nn == 0:
                rows.append({"variant": v, "feature": f, "action": "drop", "reason": "all_missing_in_df_task"})
                continue

            miss = float(df_task[f].isna().mean())
            if miss > cfg.drop_if_missing_gt:
                rows.append({"variant": v, "feature": f, "action": "drop", "reason": f"missing_gt_{cfg.drop_if_missing_gt}"})
                continue

            if cfg.drop_constant_features:
                vc = df_task[f].value_counts(dropna=True)
                if len(vc) == 1:
                    rows.append({"variant": v, "feature": f, "action": "drop", "reason": "constant_feature"})
                    continue
                top_frac = float(vc.iloc[0] / vc.sum()) if vc.sum() > 0 else 1.0
                if top_frac >= cfg.drop_near_constant_threshold:
                    rows.append({"variant": v, "feature": f, "action": "drop", "reason": f"near_constant_{top_frac:.4f}"})
                    continue

            kept.append(f)
            rows.append({"variant": v, "feature": f, "action": "keep", "reason": ""})

        out[v] = kept
    return out, pd.DataFrame(rows)

variants_safe2, allmiss_audit = drop_all_missing_high_missing_constant(df_task, variants_safe, CFG)
allmiss_audit.to_csv(ALLMISSING_AUDIT, index=False)

print("Variant sizes after global quality drop:", {k: len(v) for k, v in variants_safe2.items()})
print("Saved:", ALLMISSING_AUDIT.name)


Variant sizes after global quality drop: {'FULL': 69, 'CLINICAL': 58, 'BIOMARKERS': 11}
Saved: S2_all_missing_drop_audit.csv


## Biomarkers leakage triage + blacklist (univariate CV AUC)

In [9]:
def univariate_auc_triage(
    df_task: pd.DataFrame,
    y: pd.Series,
    feats: List[str],
    cfg: Config,
    n_splits: int,
) -> pd.DataFrame:
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=cfg.random_state)
    rows = []

    for f in feats:
        s = df_task[f]
        s2 = pd.to_numeric(s, errors="coerce") if not pd.api.types.is_numeric_dtype(s) else s.astype(float)

        nonmiss_rate = float(s2.notna().mean())
        if nonmiss_rate < cfg.triage_min_nonmissing:
            rows.append({"feature": f, "status": "skip", "reason": f"too_missing<{cfg.triage_min_nonmissing}", "auc_mean": np.nan, "auc_sd": np.nan})
            continue
        if s2.nunique(dropna=True) < cfg.triage_min_unique:
            rows.append({"feature": f, "status": "skip", "reason": f"too_constant<uniq{cfg.triage_min_unique}", "auc_mean": np.nan, "auc_sd": np.nan})
            continue

        fold_aucs = []
        for tr, te in skf.split(df_task, y.astype(int)):
            tr_s = s2.iloc[tr]
            te_s = s2.iloc[te]

            med = np.nanmedian(tr_s.to_numpy())
            te_filled = np.where(np.isnan(te_s), med, te_s)

            try:
                auc = roc_auc_score(y.iloc[te].astype(int), te_filled)
                auc = max(float(auc), 1.0 - float(auc))
                fold_aucs.append(auc)
            except Exception:
                continue

        if len(fold_aucs) == 0:
            rows.append({"feature": f, "status": "skip", "reason": "auc_failed", "auc_mean": np.nan, "auc_sd": np.nan})
        else:
            rows.append({
                "feature": f,
                "status": "ok",
                "reason": "",
                "auc_mean": float(np.mean(fold_aucs)),
                "auc_sd": float(np.std(fold_aucs)),
            })

    return pd.DataFrame(rows)

bio_feats = variants_safe2.get("BIOMARKERS", [])
triage = univariate_auc_triage(df_task, y, bio_feats, CFG, n_splits=min(5, CFG.outer_splits))
triage.to_csv(TRIAGE_PATH, index=False)

triage_ok = triage[triage["status"] == "ok"].copy().sort_values("auc_mean", ascending=False)
bio_blacklist = triage_ok.loc[triage_ok["auc_mean"] >= CFG.triage_auc_threshold, "feature"].astype(str).tolist()

pd.DataFrame(
    {"feature": bio_blacklist, "reason": [f"triage_auc_ge_{CFG.triage_auc_threshold}"] * len(bio_blacklist)}
).to_csv(BIO_BLACKLIST_PATH, index=False)

print("Saved:", TRIAGE_PATH.name, "| n_ok:", int((triage["status"]=="ok").sum()))
print("Biomarker blacklist size:", len(bio_blacklist), "| saved:", BIO_BLACKLIST_PATH.name)
display(triage_ok.head(20))


Saved: S2_biomarkers_univariate_triage.csv | n_ok: 11
Biomarker blacklist size: 1 | saved: S2_biomarkers_blacklist.csv


,feature,status,reason,auc_mean,auc_sd
1,92_kda_pro_mmp_9,ok,,0.988571,0.022857
0,72_kda_pro_mmp_2,ok,,0.847619,0.137255
3,emmprin,ok,,0.757143,0.118647
10,wyniki_klotho_pg_ml,ok,,0.749048,0.149296
9,wyniki_fgf23_pg_ml,ok,,0.698095,0.101802
4,hdl,ok,,0.672857,0.065181
2,cholesterol_calkowity_tc,ok,,0.652381,0.071333
7,non_hdl_calkowity_hdl,ok,,0.637619,0.091205
8,tnfalfa_pg_ml,ok,,0.634286,0.114182
5,il_6_pg_ml,ok,,0.624762,0.105598


## Final variants after blacklist

In [10]:
# ============================================================================
# CRITICAL METHODOLOGICAL NOTE: Differential Blacklist Application
# ============================================================================
# The biomarker blacklist (features with univariate AUC >= 0.98) is applied ONLY
# to the BIOMARKERS variant, NOT to the FULL variant. This is BY DESIGN.
#
# Rationale (per manuscript Section 5 and 6):
# - FULL variant is intended to demonstrate what happens under UNCONSTRAINED
#   feature governance — including retention of potentially leaky predictors
# - Near-perfect discrimination in FULL serves as a DIAGNOSTIC SIGNAL of
#   potential information leakage, not as evidence of clinical readiness
# - BIOMARKERS variant applies stricter governance to show "cleaner" behavior
#
# This contrast is pedagogical: it illustrates how feature governance choices
# fundamentally alter apparent model performance and its interpretation.
# ============================================================================

variants_final = dict(variants_safe2)

# Apply blacklist to BIOMARKERS only (FULL intentionally retains all features)
# FULL is designed to demonstrate behavior under unconstrained feature governance,
# serving as a leakage diagnostic signal per the methodological framework (Section 5).
variants_final["BIOMARKERS"] = [f for f in variants_final.get("BIOMARKERS", []) if f not in set(bio_blacklist)]

print("Final variant sizes:", {k: len(v) for k, v in variants_final.items()})


for vname, feats in variants_final.items():
    if len(feats) < 2:
        raise ValueError(f"Variant '{vname}' has <2 features after governance + blacklist. Cannot run nested CV.")


Final variant sizes: {'FULL': 69, 'CLINICAL': 58, 'BIOMARKERS': 10}


## Preprocessing utilities

In [11]:
def infer_feature_types(X: pd.DataFrame) -> Tuple[List[str], List[str]]:
    num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    cat_cols = [c for c in X.columns if c not in num_cols]
    return num_cols, cat_cols

def make_preprocessor(X_train: pd.DataFrame) -> ColumnTransformer:
    num_cols, cat_cols = infer_feature_types(X_train)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])


    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", ohe),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_cols),
            ("cat", cat_pipe, cat_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

def _standardize_for_sklearn(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    for col in X.columns:
        if pd.api.types.is_bool_dtype(X[col]):
            X[col] = X[col].replace({True: 1.0, False: 0.0, pd.NA: np.nan}).astype(float)
        elif pd.api.types.is_numeric_dtype(X[col]):
            X[col] = pd.to_numeric(X[col], errors="coerce")
        else:
            X[col] = X[col].astype(object).replace({pd.NA: np.nan, None: np.nan})
            X[col] = X[col].replace("nan", np.nan)
    return X


## Nested CV runner (fold-level all-missing drop audit)

In [12]:
def run_nested_cv_for_variant(
    df_task: pd.DataFrame,
    variant: str,
    features: List[str],
    y: pd.Series,
    cfg: Config,
):
    outer = StratifiedKFold(n_splits=cfg.outer_splits, shuffle=True, random_state=cfg.random_state)
    inner = StratifiedKFold(n_splits=cfg.inner_splits, shuffle=True, random_state=cfg.random_state)

    feats = [f for f in features if f in df_task.columns and f != cfg.target_col]
    X_all = _standardize_for_sklearn(df_task[feats].copy())
    y_all = y.copy().astype(int)

    X_all = X_all.reset_index(drop=True)
    y_all = y_all.reset_index(drop=True)

    results_rows, oof_rows, bestparams_rows, outer_idx_rows = [], [], [], []
    fold_drop_rows = []

    for fold_i, (tr_idx, te_idx) in enumerate(outer.split(X_all, y_all), start=1):
        X_tr, X_te = X_all.iloc[tr_idx].copy(), X_all.iloc[te_idx].copy()
        y_tr, y_te = y_all.iloc[tr_idx].copy(), y_all.iloc[te_idx].copy()

        if len(np.unique(y_tr)) < 2:
            raise ValueError(f"[{variant}] Outer fold {fold_i}: training has <2 classes. Reduce n_splits.")


        all_missing_in_train = [c for c in X_tr.columns if X_tr[c].notna().sum() == 0]
        if all_missing_in_train:
            for c in all_missing_in_train:
                fold_drop_rows.append({"variant": variant, "outer_fold": fold_i, "feature": c, "reason": "all_missing_in_train"})
            X_tr = X_tr.drop(columns=all_missing_in_train)
            X_te = X_te.drop(columns=all_missing_in_train)

        if X_tr.shape[1] < 2:
            raise ValueError(f"[{variant}] Outer fold {fold_i}: <2 usable features after dropping all-missing-in-train columns.")

        note = ""
        if len(np.unique(y_te)) < 2:
            note = "test_has_single_class_skipped_metrics"
            outer_idx_rows.append({
                "variant": variant,
                "outer_fold": fold_i,
                "train_indices": json.dumps(tr_idx.tolist()),
                "test_indices": json.dumps(te_idx.tolist()),
                "note": note,
            })
            continue

        outer_idx_rows.append({
            "variant": variant,
            "outer_fold": fold_i,
            "train_indices": json.dumps(tr_idx.tolist()),
            "test_indices": json.dumps(te_idx.tolist()),
            "note": "",
        })

        pre = make_preprocessor(X_tr)

        for model_name, space in MODEL_SPACES.items():
            pipe = Pipeline([("pre", pre), ("clf", space["estimator"])])

            grid = GridSearchCV(
                pipe,
                param_grid=space["param_grid"],
                scoring=cfg.scoring,
                cv=inner,
                n_jobs=-1,
                refit=True,
            )
            grid.fit(X_tr, y_tr)

            best = grid.best_estimator_
            prob = best.predict_proba(X_te)[:, 1]
            pred = (prob >= 0.5).astype(int)

            results_rows.append({
                "variant": variant,
                "outer_fold": fold_i,
                "model": model_name,
                "roc_auc": float(roc_auc_score(y_te, prob)),
                "auprc": float(average_precision_score(y_te, prob)),
                "accuracy": float(accuracy_score(y_te, pred)),
                "f1": float(f1_score(y_te, pred, zero_division=0)),
                "mcc": float(matthews_corrcoef(y_te, pred)),
                "brier": float(brier_score_loss(y_te, prob)),
                "n_test": int(len(y_te)),
                "pos_rate_test": float(np.mean(y_te)),
                "n_features_train": int(X_tr.shape[1]),
            })

            bestparams_rows.append({
                "variant": variant,
                "outer_fold": fold_i,
                "model": model_name,
                "best_inner_score_auc": float(grid.best_score_),
                "best_params": json.dumps(grid.best_params_),
            })

            for j, p in zip(te_idx, prob):
                oof_rows.append({
                    "variant": variant,
                    "outer_fold": fold_i,
                    "model": model_name,
                    "row_id": int(j),
                    "y_true": int(y_all.iloc[j]),
                    "y_prob": float(p),
                })

    return (
        pd.DataFrame(results_rows),
        pd.DataFrame(oof_rows),
        pd.DataFrame(bestparams_rows),
        pd.DataFrame(outer_idx_rows),
        pd.DataFrame(fold_drop_rows),
    )


## Summaries + calibration + DCA

In [13]:
def summarize_winners(results_df: pd.DataFrame) -> pd.DataFrame:
    agg = (
        results_df.groupby(["variant", "model"], as_index=False)
        .agg(
            roc_auc_mean=("roc_auc", "mean"),
            roc_auc_sd=("roc_auc", "std"),
            brier_mean=("brier", "mean"),
            brier_sd=("brier", "std"),
            auprc_mean=("auprc", "mean"),
            auprc_sd=("auprc", "std"),
        )
        .sort_values(["variant", "roc_auc_mean"], ascending=[True, False])
    )
    return agg.groupby("variant", as_index=False).head(1).reset_index(drop=True)

def decision_curve_net_benefit(y_true: np.ndarray, y_prob: np.ndarray, thresholds: Iterable[float]) -> pd.DataFrame:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    N = len(y_true)
    eps = 1e-12
    rows = []
    for pt in thresholds:
        pt = float(pt)
        if pt <= 0 or pt >= 1:
            continue
        pred = (y_prob >= pt).astype(int)
        tp = int(((pred == 1) & (y_true == 1)).sum())
        fp = int(((pred == 1) & (y_true == 0)).sum())
        nb = (tp / N) - (fp / N) * (pt / max(1 - pt, eps))
        rows.append({"threshold": pt, "net_benefit": nb, "tp": tp, "fp": fp, "N": N})
    return pd.DataFrame(rows)

def compute_oof_calibration_and_dca(oof_df: pd.DataFrame, out_dir: Path, thresholds: Iterable[float]) -> None:
    cal_rows, dca_frames, calcurve_frames = [], [], []

    for (variant, model), g in oof_df.groupby(["variant", "model"]):
        y_true = g["y_true"].to_numpy()
        y_prob = g["y_prob"].to_numpy()

        cal_rows.append({
            "variant": variant,
            "model": model,
            "oof_roc_auc": float(roc_auc_score(y_true, y_prob)),
            "oof_auprc": float(average_precision_score(y_true, y_prob)),
            "oof_brier": float(brier_score_loss(y_true, y_prob)),
            "n": int(len(y_true)),
            "pos_rate": float(np.mean(y_true)),
        })

        frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy="quantile")
        calcurve_frames.append(pd.DataFrame({
            "variant": variant,
            "model": model,
            "bin": np.arange(1, len(frac_pos) + 1),
            "mean_predicted_prob": mean_pred,
            "fraction_of_positives": frac_pos
        }))

        dca = decision_curve_net_benefit(y_true, y_prob, thresholds)
        dca["variant"] = variant
        dca["model"] = model
        dca_frames.append(dca)

    pd.DataFrame(cal_rows).to_csv(out_dir / "oof_calibration_summary.csv", index=False)
    pd.concat(calcurve_frames, ignore_index=True).to_csv(out_dir / "oof_calibration_curve.csv", index=False)
    pd.concat(dca_frames, ignore_index=True).to_csv(out_dir / "oof_dca_curves.csv", index=False)
    print("Saved calibration + DCA outputs.")


## Governance summary + run all variants + save outputs

In [14]:
def governance_summary(variants_before: Dict[str, List[str]], variants_after: Dict[str, List[str]]) -> pd.DataFrame:
    rows = []
    for v in variants_before:
        before = set(variants_before[v])
        after = set(variants_after.get(v, []))
        rows.append({
            "variant": v,
            "n_before": len(before),
            "n_after": len(after),
            "n_dropped": len(before - after),
            "n_kept": len(after),
        })
    return pd.DataFrame(rows)

gs = governance_summary(variants_s1, variants_final)
gs.to_csv(GOV_SUMMARY, index=False)
print("Saved:", GOV_SUMMARY.name)
display(gs)

all_results, all_oof, all_bestparams, all_outer_idx, all_fold_drops = [], [], [], [], []

for variant_name, feat_list in variants_final.items():
    pd.DataFrame({"variant": [variant_name]*len(feat_list), "feature": feat_list}).to_csv(
        OUT_DIR / f"features_used_{variant_name}.csv", index=False
    )

    res_df, oof_df, bp_df, oi_df, fold_drop_df = run_nested_cv_for_variant(
        df_task, variant_name, feat_list, y, CFG
    )

    all_results.append(res_df)
    all_oof.append(oof_df)
    all_bestparams.append(bp_df)
    all_outer_idx.append(oi_df)
    all_fold_drops.append(fold_drop_df)

results_df = pd.concat(all_results, ignore_index=True)
oof_df = pd.concat(all_oof, ignore_index=True)
bestparams_df = pd.concat(all_bestparams, ignore_index=True)
outer_idx_df = pd.concat(all_outer_idx, ignore_index=True)
fold_drop_all = pd.concat(all_fold_drops, ignore_index=True) if all_fold_drops else pd.DataFrame()

results_df.to_csv(OUT_DIR / "nestedcv_results_all_variants.csv", index=False)
oof_df.to_csv(OUT_DIR / "nestedcv_oof_probs_all_variants.csv", index=False)
bestparams_df.to_csv(OUT_DIR / "nestedcv_bestparams_all_variants.csv", index=False)
outer_idx_df.to_csv(OUT_DIR / "outer_folds_indices.csv", index=False)
fold_drop_all.to_csv(FOLD_DROP_AUDIT, index=False)

winners_df = summarize_winners(results_df)
winners_df.to_csv(OUT_DIR / "winner_model_by_variant.csv", index=False)

compute_oof_calibration_and_dca(oof_df, OUT_DIR, CFG.dca_thresholds)

print("\n=== Winners (mean outer-fold ROC AUC) ===")
print(winners_df.to_string(index=False))
print("\nSaved outputs to:", OUT_DIR.resolve())
print("Saved fold drop audit:", FOLD_DROP_AUDIT.name)


Saved: S2_governance_summary.csv


,variant,n_before,n_after,n_dropped,n_kept
0,FULL,73,69,4,69
1,CLINICAL,62,58,4,58
2,BIOMARKERS,11,10,1,10


Saved calibration + DCA outputs.

=== Winners (mean outer-fold ROC AUC) ===
   variant model  roc_auc_mean  roc_auc_sd  brier_mean  brier_sd  auprc_mean  auprc_sd
BIOMARKERS    RF      0.949524    0.050575    0.110719  0.030843    0.934000  0.074027
  CLINICAL    RF      0.594286    0.115441    0.236608  0.009888    0.616436  0.156740
      FULL    RF      0.994286    0.012778    0.077564  0.057060    0.993333  0.014907

Saved outputs to: /content/results/S2_nestedcv
Saved fold drop audit: S2_fold_allmissing_drop_audit.csv


## Save core artifacts

In [15]:
CORE_RESULTS_PATH = OUT_DIR / "nestedcv_results_all_variants.csv"
CORE_OOF_PATH = OUT_DIR / "nestedcv_oof_probs_all_variants.csv"
CORE_BESTPARAMS_PATH = OUT_DIR / "nestedcv_bestparams_all_variants.csv"
CORE_OUTERIDX_PATH = OUT_DIR / "outer_folds_indices.csv"
CORE_WINNERS_PATH = OUT_DIR / "winner_model_by_variant.csv"

results_df.to_csv(CORE_RESULTS_PATH, index=False)
oof_df.to_csv(CORE_OOF_PATH, index=False)
bestparams_df.to_csv(CORE_BESTPARAMS_PATH, index=False)
outer_idx_df.to_csv(CORE_OUTERIDX_PATH, index=False)
winners_df.to_csv(CORE_WINNERS_PATH, index=False)

print("Saved core artifacts:")
for p in [CORE_RESULTS_PATH, CORE_OOF_PATH, CORE_BESTPARAMS_PATH, CORE_OUTERIDX_PATH, CORE_WINNERS_PATH]:
    print(" -", p.name)


Saved core artifacts:
 - nestedcv_results_all_variants.csv
 - nestedcv_oof_probs_all_variants.csv
 - nestedcv_bestparams_all_variants.csv
 - outer_folds_indices.csv
 - winner_model_by_variant.csv


## Save features_used for ALL variants

In [16]:


for variant_name, feat_list in variants_final.items():
    feat_path = OUT_DIR / f"features_used_{variant_name}.csv"
    pd.DataFrame({"variant": [variant_name]*len(feat_list), "feature": feat_list}).to_csv(feat_path, index=False)
    print("Saved:", feat_path.name, "| n_features:", len(feat_list))


Saved: features_used_FULL.csv | n_features: 69
Saved: features_used_CLINICAL.csv | n_features: 58
Saved: features_used_BIOMARKERS.csv | n_features: 10


## Governance breakdown (firewall vs missingness/constant vs triage blacklist)

In [17]:

def _count_by_reason(df: pd.DataFrame, reason_col: str, only_dropped: bool = True, status_col: Optional[str] = None):
    d = df.copy()
    if only_dropped and status_col is not None:
        d = d[d[status_col].isin(["dropped", "drop"])]
    return d.groupby(["variant", reason_col], as_index=False).size().rename(columns={"size": "n"})


firewall_drops = firewall_audit.copy()

fw_counts = _count_by_reason(firewall_drops, reason_col="reason", only_dropped=True, status_col="status")
fw_counts["stage"] = "S2_firewall_regex"


miss_drops = allmiss_audit.copy()

miss_counts = _count_by_reason(miss_drops, reason_col="reason", only_dropped=True, status_col="action")
miss_counts["stage"] = "S2_missingness_constant"


bl_df = pd.DataFrame({"feature": bio_blacklist})
if not bl_df.empty:
    bl_df["variant"] = "BIOMARKERS"
    bl_df["reason"] = f"triage_auc_ge_{CFG.triage_auc_threshold}"
    bl_counts = bl_df.groupby(["variant", "reason"], as_index=False).size().rename(columns={"size": "n"})
    bl_counts["stage"] = "S2_biomarkers_triage_blacklist"
else:
    bl_counts = pd.DataFrame(columns=["variant", "reason", "n", "stage"])

breakdown = pd.concat([fw_counts, miss_counts, bl_counts], ignore_index=True)

BREAKDOWN_PATH = OUT_DIR / "S2_governance_breakdown.csv"
breakdown.to_csv(BREAKDOWN_PATH, index=False)

print("Saved:", BREAKDOWN_PATH.name)
display(breakdown.sort_values(["variant", "stage", "n"], ascending=[True, True, False]).head(50))


Saved: S2_governance_breakdown.csv


,variant,reason,n,stage
4,BIOMARKERS,triage_auc_ge_0.98,1,S2_biomarkers_triage_blacklist
0,CLINICAL,all_missing_in_df_task,3,S2_missingness_constant
1,CLINICAL,missing_gt_0.95,1,S2_missingness_constant
2,FULL,all_missing_in_df_task,3,S2_missingness_constant
3,FULL,missing_gt_0.95,1,S2_missingness_constant


## Hard sanity checks for leakage-like symptoms & structural correctness

In [18]:


def assert_no_blacklisted_biomarkers(variants_final: Dict[str, List[str]], bio_blacklist: List[str]):
    if not bio_blacklist:
        return
    bad = set(variants_final.get("BIOMARKERS", [])).intersection(set(bio_blacklist))
    if bad:
        raise ValueError(f"[S2] Blacklisted biomarker(s) still present in BIOMARKERS variant: {sorted(list(bad))[:20]}")

def assert_oof_integrity(oof_df: pd.DataFrame):
    required = {"variant", "outer_fold", "model", "row_id", "y_true", "y_prob"}
    missing = required - set(oof_df.columns)
    if missing:
        raise ValueError(f"[S2] oof_df missing columns: {missing}")

    if oof_df["y_prob"].isna().any():
        n = int(oof_df["y_prob"].isna().sum())
        raise ValueError(f"[S2] oof_df contains NA probabilities: n={n}")


    dup = oof_df.duplicated(subset=["variant", "model", "row_id"], keep=False)
    if dup.any():
        ex = oof_df.loc[dup, ["variant","model","row_id","outer_fold"]].head(20)
        raise ValueError("[S2] Duplicate OOF rows detected for (variant, model, row_id). Example:\n" + ex.to_string(index=False))

def assert_outeridx_integrity(outer_idx_df: pd.DataFrame):
    required = {"variant", "outer_fold", "train_indices", "test_indices", "note"}
    missing = required - set(outer_idx_df.columns)
    if missing:
        raise ValueError(f"[S2] outer_idx_df missing columns: {missing}")


    for v in outer_idx_df["variant"].unique():
        nfold = outer_idx_df[outer_idx_df["variant"] == v]["outer_fold"].nunique()
        if nfold < 3:
            raise ValueError(f"[S2] Too few outer folds recorded for variant={v}: nfold={nfold}")

def leakage_like_flag_summary(results_df: pd.DataFrame, oof_df: pd.DataFrame):

    suspicious = results_df[(results_df["roc_auc"] >= 0.999) | (results_df["auprc"] >= 0.999)].copy()

    oof_sus = []
    for (v,m), g in oof_df.groupby(["variant","model"]):
        try:
            auc = roc_auc_score(g["y_true"].to_numpy(), g["y_prob"].to_numpy())
            auprc = average_precision_score(g["y_true"].to_numpy(), g["y_prob"].to_numpy())
            if (auc >= 0.995) or (auprc >= 0.995):
                oof_sus.append({"variant": v, "model": m, "oof_auc": float(auc), "oof_auprc": float(auprc), "n": int(len(g))})
        except Exception:
            continue
    oof_sus_df = pd.DataFrame(oof_sus).sort_values(["oof_auc","oof_auprc"], ascending=False) if oof_sus else pd.DataFrame()

    return suspicious, oof_sus_df

assert_no_blacklisted_biomarkers(variants_final, bio_blacklist)
assert_oof_integrity(oof_df)
assert_outeridx_integrity(outer_idx_df)

sus_fold, sus_oof = leakage_like_flag_summary(results_df, oof_df)

SUS_PATH = OUT_DIR / "S2_leakage_like_perf_flags.csv"
pd.concat([
    sus_fold.assign(flag_source="outer_fold_metrics"),
    sus_oof.assign(flag_source="oof_metrics"),
], ignore_index=True).to_csv(SUS_PATH, index=False)

print("Saved:", SUS_PATH.name)
print("Leakage-like perfect fold metrics:", len(sus_fold))
print("Leakage-like perfect OOF metrics:", len(sus_oof))
if len(sus_oof):
    display(sus_oof.head(20))


Saved: S2_leakage_like_perf_flags.csv
Leakage-like perfect fold metrics: 15
Leakage-like perfect OOF metrics: 1


,variant,model,oof_auc,oof_auprc,n
0,FULL,RF,0.99875,0.998462,57


## Minimal manifest for S3–S11 (one file to validate completeness)

In [19]:


def file_info(path: Path) -> Dict[str, Any]:
    return {
        "name": path.name,
        "exists": path.exists(),
        "bytes": int(path.stat().st_size) if path.exists() else None,
    }

manifest = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "task": {"labels_allowed": list(CFG.task_labels_allowed), "positive_label": CFG.positive_label},
    "n_df_task": int(len(df_task)),
    "pos_rate": float(np.mean(y.astype(int))),
    "variants_final_sizes": {k: int(len(v)) for k, v in variants_final.items()},
    "artifacts": {
        "nestedcv_results_all_variants": file_info(CORE_RESULTS_PATH),
        "nestedcv_oof_probs_all_variants": file_info(CORE_OOF_PATH),
        "nestedcv_bestparams_all_variants": file_info(CORE_BESTPARAMS_PATH),
        "outer_folds_indices": file_info(CORE_OUTERIDX_PATH),
        "winner_model_by_variant": file_info(CORE_WINNERS_PATH),
        "oof_calibration_summary": file_info(OUT_DIR / "oof_calibration_summary.csv"),
        "oof_calibration_curve": file_info(OUT_DIR / "oof_calibration_curve.csv"),
        "oof_dca_curves": file_info(OUT_DIR / "oof_dca_curves.csv"),
        "governance_summary": file_info(GOV_SUMMARY),
        "governance_breakdown": file_info(OUT_DIR / "S2_governance_breakdown.csv"),
        "leakage_like_flags": file_info(OUT_DIR / "S2_leakage_like_perf_flags.csv"),
        "biomarkers_triage": file_info(TRIAGE_PATH),
        "biomarkers_blacklist": file_info(BIO_BLACKLIST_PATH),
        "all_missing_drop_audit": file_info(ALLMISSING_AUDIT),
        "fold_allmissing_drop_audit": file_info(FOLD_DROP_AUDIT),
    },
}

MANIFEST_PATH = OUT_DIR / "S2_ready_for_S3_S11_manifest.json"
save_json(manifest, MANIFEST_PATH)

print("Saved:", MANIFEST_PATH.name)
print("Artifacts missing (exists==False):")
missing = [k for k,v in manifest["artifacts"].items() if v["exists"] is False]
print(missing if missing else "None ")


Saved: S2_ready_for_S3_S11_manifest.json
Artifacts missing (exists==False):
None 


## Loading results + sanity-check

In [20]:

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    roc_curve,
    precision_recall_curve,
)

OUT_DIR = Path("/content/results/S2_nestedcv")

RES_PATH = OUT_DIR / "nestedcv_results_all_variants.csv"
OOF_PATH = OUT_DIR / "nestedcv_oof_probs_all_variants.csv"
WIN_PATH = OUT_DIR / "winner_model_by_variant.csv"

assert RES_PATH.exists(), f"Missing: {RES_PATH}"
assert OOF_PATH.exists(), f"Missing: {OOF_PATH}"
assert WIN_PATH.exists(), f"Missing: {WIN_PATH}"

results_df = pd.read_csv(RES_PATH)
oof_df = pd.read_csv(OOF_PATH)
winners_df = pd.read_csv(WIN_PATH)

print("Loaded:")
print("- results_df:", results_df.shape)
print("- oof_df:", oof_df.shape)
print("- winners_df:", winners_df.shape)


needed_cols_results = {"variant","outer_fold","model","roc_auc","auprc","brier"}
needed_cols_oof = {"variant","outer_fold","model","row_id","y_true","y_prob"}
needed_cols_win = {"variant","model"}

assert needed_cols_results.issubset(results_df.columns), f"results_df missing: {needed_cols_results - set(results_df.columns)}"
assert needed_cols_oof.issubset(oof_df.columns), f"oof_df missing: {needed_cols_oof - set(oof_df.columns)}"
assert needed_cols_win.issubset(winners_df.columns), f"winners_df missing: {needed_cols_win - set(winners_df.columns)}"


dup = oof_df.groupby(["variant","model","row_id"]).size()
assert (dup == 1).all(), "OOF has duplicate predictions for some variant/model/row_id."


n_per = oof_df.groupby(["variant","model"])["row_id"].nunique().reset_index(name="n_unique_rows")
print("\nOOF unique rows per (variant, model):")
display(n_per)

print("\nSanity OK ")


Loaded:
- results_df: (60, 12)
- oof_df: (684, 6)
- winners_df: (3, 8)

OOF unique rows per (variant, model):


,variant,model,n_unique_rows
0,BIOMARKERS,HGB,57
1,BIOMARKERS,LinearSVM,57
2,BIOMARKERS,LogReg,57
3,BIOMARKERS,RF,57
4,CLINICAL,HGB,57
5,CLINICAL,LinearSVM,57
6,CLINICAL,LogReg,57
7,CLINICAL,RF,57
8,FULL,HGB,57
9,FULL,LinearSVM,57



Sanity OK 


## Fold-metrics + 95% CI

In [21]:

from math import sqrt

def mean_ci95(values: np.ndarray) -> tuple[float, float, float]:

    vals = np.asarray(values, dtype=float)
    vals = vals[~np.isnan(vals)]
    n = len(vals)
    if n == 0:
        return (np.nan, np.nan, np.nan)
    m = float(np.mean(vals))
    sd = float(np.std(vals, ddof=1)) if n > 1 else 0.0
    se = sd / sqrt(n) if n > 0 else np.nan

    lo = m - 1.96 * se
    hi = m + 1.96 * se
    return (m, lo, hi)

rows = []
for (variant, model), g in results_df.groupby(["variant","model"]):
    for metric in ["roc_auc","auprc","brier","accuracy","f1","mcc"]:
        if metric not in g.columns:
            continue
        m, lo, hi = mean_ci95(g[metric].to_numpy())
        rows.append({
            "variant": variant,
            "model": model,
            "metric": metric,
            "mean": m,
            "ci95_lo": lo,
            "ci95_hi": hi,
            "sd": float(np.std(g[metric].to_numpy(), ddof=1)) if len(g) > 1 else 0.0,
            "n_folds": int(g["outer_fold"].nunique()),
        })

fold_metrics_summary = pd.DataFrame(rows).sort_values(["variant","metric","model"])
out_path = OUT_DIR / "fold_metrics_summary_by_variant_model.csv"
fold_metrics_summary.to_csv(out_path, index=False)
print("Saved:", out_path.name)
display(fold_metrics_summary.head(30))


Saved: fold_metrics_summary_by_variant_model.csv


,variant,model,metric,mean,ci95_lo,ci95_hi,sd,n_folds
3,BIOMARKERS,HGB,accuracy,0.860606,0.821433,0.899779,0.044691,5
9,BIOMARKERS,LinearSVM,accuracy,0.754545,0.620038,0.889053,0.153454,5
15,BIOMARKERS,LogReg,accuracy,0.721212,0.611756,0.830669,0.124874,5
21,BIOMARKERS,RF,accuracy,0.878788,0.811723,0.945853,0.076511,5
1,BIOMARKERS,HGB,auprc,0.830000,0.760889,0.899111,0.078846,5
7,BIOMARKERS,LinearSVM,auprc,0.795984,0.675310,0.916658,0.137671,5
13,BIOMARKERS,LogReg,auprc,0.729270,0.631781,0.826759,0.111220,5
19,BIOMARKERS,RF,auprc,0.934000,0.869112,0.998888,0.074027,5
2,BIOMARKERS,HGB,brier,0.117966,0.100522,0.135410,0.019901,5
8,BIOMARKERS,LinearSVM,brier,0.164683,0.115004,0.214362,0.056676,5


## Bootstrap CI on OOF for winners

In [22]:


def stratified_bootstrap_indices(y: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    y = np.asarray(y).astype(int)
    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]
    b0 = rng.choice(idx0, size=len(idx0), replace=True)
    b1 = rng.choice(idx1, size=len(idx1), replace=True)
    b = np.concatenate([b0, b1])
    rng.shuffle(b)
    return b

def bootstrap_ci(y_true: np.ndarray, y_prob: np.ndarray, metric_fn, n_boot: int = 2000, seed: int = 42):
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n_boot):
        b = stratified_bootstrap_indices(y_true, rng)
        try:
            vals.append(metric_fn(y_true[b], y_prob[b]))
        except Exception:
            continue
    vals = np.asarray(vals, dtype=float)
    if len(vals) == 0:
        return (np.nan, np.nan, np.nan)
    return (float(np.mean(vals)), float(np.quantile(vals, 0.025)), float(np.quantile(vals, 0.975)))

def m_roc(y, p): return roc_auc_score(y, p)
def m_pr(y, p):  return average_precision_score(y, p)
def m_brier(y, p): return brier_score_loss(y, p)

winner_pairs = winners_df[["variant","model"]].drop_duplicates()

rows = []
for _, r in winner_pairs.iterrows():
    variant = r["variant"]
    model = r["model"]
    g = oof_df[(oof_df["variant"] == variant) & (oof_df["model"] == model)].copy()

    y_true = g["y_true"].to_numpy().astype(int)
    y_prob = g["y_prob"].to_numpy().astype(float)

    roc_mean, roc_lo, roc_hi = bootstrap_ci(y_true, y_prob, m_roc, n_boot=3000, seed=42)
    pr_mean, pr_lo, pr_hi = bootstrap_ci(y_true, y_prob, m_pr, n_boot=3000, seed=43)
    br_mean, br_lo, br_hi = bootstrap_ci(y_true, y_prob, m_brier, n_boot=3000, seed=44)

    rows.append({
        "variant": variant,
        "model": model,
        "oof_n": int(len(y_true)),
        "pos_rate": float(np.mean(y_true)),
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "roc_auc_boot_mean": roc_mean,
        "roc_auc_ci95_lo": roc_lo,
        "roc_auc_ci95_hi": roc_hi,
        "auprc": float(average_precision_score(y_true, y_prob)),
        "auprc_boot_mean": pr_mean,
        "auprc_ci95_lo": pr_lo,
        "auprc_ci95_hi": pr_hi,
        "brier": float(brier_score_loss(y_true, y_prob)),
        "brier_boot_mean": br_mean,
        "brier_ci95_lo": br_lo,
        "brier_ci95_hi": br_hi,
    })

winner_oof_ci = pd.DataFrame(rows).sort_values(["variant"])
out_path = OUT_DIR / "winner_oof_metrics_with_bootstrap_ci.csv"
winner_oof_ci.to_csv(out_path, index=False)
print("Saved:", out_path.name)
display(winner_oof_ci)


Saved: winner_oof_metrics_with_bootstrap_ci.csv


,variant,model,oof_n,pos_rate,roc_auc,roc_auc_boot_mean,roc_auc_ci95_lo,roc_auc_ci95_hi,auprc,auprc_boot_mean,auprc_ci95_lo,auprc_ci95_hi,brier,brier_boot_mean,brier_ci95_lo,brier_ci95_hi
0,BIOMARKERS,RF,57,0.438596,0.93000,0.931194,0.853719,0.986281,0.892808,0.899228,0.790555,0.985762,0.111469,0.111276,0.077886,0.148542
1,CLINICAL,RF,57,0.438596,0.60250,0.601490,0.446250,0.745031,0.579670,0.593027,0.452652,0.732252,0.236464,0.236259,0.217163,0.256173
2,FULL,RF,57,0.438596,0.99875,0.998740,0.992500,1.000000,0.998462,0.998595,0.991429,1.000000,0.077729,0.077501,0.059609,0.096010


## ROC/PR points for winners

In [23]:
from sklearn.metrics import roc_curve, precision_recall_curve

roc_frames = []
pr_frames = []

for _, r in winner_pairs.iterrows():
    variant = r["variant"]
    model = r["model"]
    g = oof_df[(oof_df["variant"] == variant) & (oof_df["model"] == model)].copy()

    y_true = g["y_true"].to_numpy().astype(int)
    y_prob = g["y_prob"].to_numpy().astype(float)

    fpr, tpr, thr = roc_curve(y_true, y_prob)
    roc_frames.append(pd.DataFrame({
        "variant": variant,
        "model": model,
        "fpr": fpr,
        "tpr": tpr,
        "threshold": thr,
    }))

    prec, rec, thr2 = precision_recall_curve(y_true, y_prob)
    pr_frames.append(pd.DataFrame({
        "variant": variant,
        "model": model,
        "precision": prec,
        "recall": rec,
        "threshold": np.r_[thr2, np.nan],
    }))

roc_points = pd.concat(roc_frames, ignore_index=True)
pr_points = pd.concat(pr_frames, ignore_index=True)

roc_path = OUT_DIR / "winner_oof_roc_curve_points.csv"
pr_path  = OUT_DIR / "winner_oof_pr_curve_points.csv"

roc_points.to_csv(roc_path, index=False)
pr_points.to_csv(pr_path, index=False)

print("Saved:", roc_path.name, "and", pr_path.name)
display(roc_points.head(10))
display(pr_points.head(10))


Saved: winner_oof_roc_curve_points.csv and winner_oof_pr_curve_points.csv


,variant,model,fpr,tpr,threshold
0,BIOMARKERS,RF,0.00000,0.00,inf
1,BIOMARKERS,RF,0.00000,0.04,0.943760
2,BIOMARKERS,RF,0.00000,0.20,0.864000
3,BIOMARKERS,RF,0.00000,0.28,0.826000
4,BIOMARKERS,RF,0.03125,0.28,0.770000
5,BIOMARKERS,RF,0.03125,0.36,0.747653
6,BIOMARKERS,RF,0.06250,0.36,0.742972
7,BIOMARKERS,RF,0.06250,0.68,0.644888
8,BIOMARKERS,RF,0.12500,0.68,0.641054
9,BIOMARKERS,RF,0.12500,0.96,0.572000


,variant,model,precision,recall,threshold
0,BIOMARKERS,RF,0.438596,1.0,0.008000
1,BIOMARKERS,RF,0.446429,1.0,0.012000
2,BIOMARKERS,RF,0.454545,1.0,0.024000
3,BIOMARKERS,RF,0.462963,1.0,0.034000
4,BIOMARKERS,RF,0.471698,1.0,0.048000
5,BIOMARKERS,RF,0.480769,1.0,0.085194
6,BIOMARKERS,RF,0.490196,1.0,0.092524
7,BIOMARKERS,RF,0.500000,1.0,0.096000
8,BIOMARKERS,RF,0.510204,1.0,0.110000
9,BIOMARKERS,RF,0.531915,1.0,0.118976


## Manifest + sha256

In [24]:

import hashlib
import json

def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


expected = [
    "package_versions.json",
    "S2_config_snapshot.json",
    "S2_leakage_sanity_flags.csv",
    "S2_all_missing_drop_audit.csv",
    "S2_biomarkers_univariate_triage.csv",
    "S2_biomarkers_blacklist.csv",
    "S2_governance_summary.csv",
    "features_used_FULL.csv",
    "features_used_CLINICAL.csv",
    "features_used_BIOMARKERS.csv",
    "nestedcv_results_all_variants.csv",
    "nestedcv_oof_probs_all_variants.csv",
    "nestedcv_bestparams_all_variants.csv",
    "outer_folds_indices.csv",
    "winner_model_by_variant.csv",
    "oof_calibration_summary.csv",
    "oof_calibration_curve.csv",
    "oof_dca_curves.csv",
    "fold_metrics_summary_by_variant_model.csv",
    "winner_oof_metrics_with_bootstrap_ci.csv",
    "winner_oof_roc_curve_points.csv",
    "winner_oof_pr_curve_points.csv",
]

manifest = []
for name in expected:
    p = OUT_DIR / name
    manifest.append({
        "file": name,
        "exists": bool(p.exists()),
        "bytes": int(p.stat().st_size) if p.exists() else None,
        "sha256": sha256_file(p) if p.exists() else None,
    })

manifest_path = OUT_DIR / "S2_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print("Saved:", manifest_path.name)

missing = [m["file"] for m in manifest if not m["exists"]]
print("Missing files:", missing if missing else "None ")


Saved: S2_manifest.json
Missing files: None 
